# Транскрибация аудиозаписей

Скрипт позволяет достаточно быстро транскрибировать конвейер из нескольких десятков аудиофайлов. Качество транскрибации хорошее. 

env: << рекомендуется под задачу транскрибации создать отдельное ядро python >>

Транскрибатор: https://www.assemblyai.com/

1) Аудиозаписи сохраняем в отдельную папку
2) Сооздаём книгу table.xlsx. У меня она сразу с таблицей рекрутинга (лист table), откуда скрипт забирает ФИО респондента для последующей диаризации. 
3) В переменную respondent прописываем номер респондента из раблицы рекрутинга. Под этим же номером сохранено видео. 
4) Результат транскрибации сохраняется в виде Excel таблицы на отдельном листе книги table.xlsx (наименование листа формируется через номер и ФИО респондента) со следующей структурой: № строки, Время начала фразы / start_time, Время окончания фразы / end_time, Говорящий / speaker, Фраза / text. Такая таблица удобна для последующей обработки и тематического кодирования. 

In [ ]:
pip install requests

In [1]:
import assemblyai as aai
import os
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')
import os
import json
from datetime import timedelta
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import PatternFill
from openpyxl.utils.dataframe import dataframe_to_rows
import xlsxwriter
import re

## 1. Подготовительный этап

In [ ]:
# Загрузка ключа
aai.settings.api_key = "сюда вписываем свой API ключ от assemblyai"

In [3]:
# принудительно назначаем номер респондента
respondent = 9

In [ ]:
# Загружаем таблицу рекрутинга
TABLE_PATH = r"C:\\Вписываем путь до таблицы\\table.xlsx"

In [5]:
# Загружаем таблицу
table = pd.read_excel(TABLE_PATH, sheet_name="table")
table.head()

,№,date,guest
0,1,2026-06-22,Антон Петров
1,2,2026-06-24,Игорь Иванов
2,3,NaT,NaN
3,4,NaT,NaN
4,5,NaT,NaN


In [6]:
# ============================================
# 2. ФУНКЦИЯ ДЛЯ ПОЛУЧЕНИЯ ДАННЫХ РЕСПОНДЕНТА
# ============================================

def get_respondent_data(respondent_number, df):
    """
    Получает данные респондента по его номеру в таблице.
    
    Args:
        respondent_number (int): Номер респондента (строка в таблице)
        df (DataFrame): DataFrame с таблицей
    
    Returns:
        dict: Словарь с данными респондента
    """
    
    # Проверяем, что номер существует в таблице
    if respondent_number > len(df):
        print(f"❌ Ошибка: Номер {respondent_number} превышает количество строк в таблице ({len(df)})")
        return None
    
    # Индексация в pandas начинается с 0, поэтому номер - 1
    row_index = respondent_number - 1
    
    # Получаем данные из строки
    row = df.iloc[row_index]
    
    # Проверяем, что данные не пустые
    if pd.isna(row['guest']) or pd.isna(row['date']):
        print(f"⚠️  Внимание: Для респондента №{respondent_number} отсутствуют данные")
        print(f"   Дата: {row['date'] if not pd.isna(row['date']) else 'Нет данных'}")
        print(f"   Имя: {row['guest'] if not pd.isna(row['guest']) else 'Нет данных'}")
        return None
    
    # Формируем результат в словарь
    result = {
        'respondent_number': respondent_number,
        'date': row['date'].strftime('%Y-%m-%d') if hasattr(row['date'], 'strftime') else str(row['date']),
        'guest': row['guest']
    }
    
    return result

In [7]:
resp_data = get_respondent_data(respondent, table)
resp_data

{'respondent_number': 9, 'date': '2026-06-25', 'guest': 'Игорь Васильев'}

In [ ]:
# Укажите путь к вашему большому видеофайлу
FILE_PATH = f'C:\\Вписываем стандартный путь до аудиофайла\\eco_{respondent}_aud.m4a'

In [9]:
# Создаем объект транскрибатора
transcriber = aai.Transcriber()

In [ ]:
# Шаг 1: Загружаем файл на сервер AssemblyAI
# Этот метод умеет работать с большими файлами
print(f"Загрузка файла '{FILE_PATH}'...")
upload_url = transcriber.upload_file(FILE_PATH)
print(f"Файл загружен, получен URL: {upload_url}")

In [ ]:
# ============================================
# 2. КОНФИГУРАЦИЯ
# ============================================

# Шаг 2: Настраиваем параметры транскрибации
# Явно указываем русский язык и включаем диаризацию
config = aai.TranscriptionConfig(
    language_code="ru",
    speaker_labels=True,
    speech_understanding={
        "request": {
            "speaker_identification": {
                "speaker_type": "role",
                "speakers": [
                    {"role": "Эльвира", "description": "Интервьюер"}, # Вписываем имя интервьюера в поле role
                    {"role": f"{resp_data['guest']}", "description": "Респондент"}
                ]
            }
        }
    }
)

## Транскрибация

In [ ]:
# Шаг 3: Запускаем транскрибацию, используя URL загруженного файла
print("Запуск транскрибации...")
transcript = transcriber.transcribe(upload_url, config=config)

for utterance in transcript.utterances:
    print(f"{utterance.speaker}: {utterance.text}")

# Шаг 4: Обрабатываем результат
if transcript.status == aai.TranscriptStatus.error:
    print(f"Произошла ошибка: {transcript.error}")
else:
    print("Транскрибация успешно завершена!")
    print("\n--- Текст с метками говорящих ---")
    for utterance in transcript.utterances:
       print(f"Говорящий {utterance.speaker}: {utterance.text}")

In [13]:
def format_time(ms):
    """Конвертирует миллисекунды в формат ЧЧ:ММ:СС (без миллисекунд)"""
    seconds = ms // 1000
    hours, seconds = divmod(seconds, 3600)
    minutes, seconds = divmod(seconds, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

## Сохраняем результат

In [ ]:
# ============================================
# 4. СОХРАНЕНИЕ РЕЗУЛЬТАТОВ В ВИДЕ ТАБЛИЦЫ EXCEL
# ============================================

if transcript.status == aai.TranscriptStatus.error:
    print(f"Произошла ошибка: {transcript.error}")
else:
    print("Транскрибация успешно завершена!")
    
    # Создаем список для данных
    data = []
    
    # Заполняем данные из транскрибации
    for idx, utterance in enumerate(transcript.utterances, start=1):
        start_time = format_time(utterance.start)
        end_time = format_time(utterance.end)
        speaker = utterance.speaker
        text = utterance.text
        
        data.append({
            "№": idx,
            "start_time": start_time,
            "end_time": end_time,
            "speaker": speaker,
            "text": text
        })
    
    # Создаем DataFrame
    df = pd.DataFrame(data)
    
    # Сохраняем в Excel
    excel_path = "C:\\Вписываем стандартный путь до таблицы\\table.xlsx"
    # df.to_excel(excel_path, index=False, sheet_name="02_trans")

    with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        df.to_excel(writer, sheet_name=f"{respondent}_{resp_data['guest']}", index=False)
    
    print(f"✅ Excel-таблица сохранена в {excel_path}")
    print(f"📊 Всего фраз: {len(data)}")
    
    # Показываем первые 5 строк в консоли для预览
    print("\n📝 Первые 5 строк таблицы:")
    print(df.head().to_string(index=False))
    
    # Также сохраняем JSON (опционально)
    structured_data = {
        "text": transcript.text,
        "audio_duration": transcript.audio_duration,
        "utterances": [
            {
                "speaker": u.speaker,
                "start": u.start,
                "end": u.end,
                "text": u.text
            }
            for u in transcript.utterances
        ]
    }
    
    # Код добавит в файл table.xlsx лист с транскриптом текущего респондента
    with open(f"C:\\Вписываем путь до папки выгрузки\\output\\transcript_str_{respondent}.json", "w", encoding="utf-8") as f:
        json.dump(structured_data, f, ensure_ascii=False, indent=2)
    
    print(f"✅ Структурированный JSON сохранен в output/transcript_str_{respondent}.json")

In [ ]:
# ============================================
# 4. СОХРАНЕНИЕ РЕЗУЛЬТАТОВ В ВИДЕ ТЕКСТА
# ============================================

if transcript.status == aai.TranscriptStatus.error:
    print(f"Произошла ошибка: {transcript.error}")
else:
    print("Транскрибация успешно завершена!")
    
    # Сохраняем с таймкодами в читаемом формате
    with open("C:\\Вписываем путь до папки выгрузки\\output\\transcript_2.txt", "w", encoding="utf-8") as f:
        f.write("=" * 60 + "\n")
        f.write("🎤 РАСШИФРОВКА С ТАЙМКОДАМИ И ГОВОРЯЩИМИ\n")
        f.write("=" * 60 + "\n\n")
        
        for utterance in transcript.utterances:
            start_time = format_time(utterance.start)
            end_time = format_time(utterance.end)
            speaker = utterance.speaker
            text = utterance.text
            
            f.write(f"[{start_time} - {end_time}] ")
            f.write(f"Говорящий {speaker}:\n")
            f.write(f"  {text}\n\n")
    
    print("✅ Подробный текст с таймкодами сохранен в output/transcript_2.txt")
    
    # Сохраняем JSON с дополнительной информацией
    # Собираем структурированные данные
    structured_data = {
        "text": transcript.text,
        "language": "ru",
        "audio_duration": transcript.audio_duration,
        "utterances": [
            {
                "speaker": u.speaker,
                "start": u.start,
                "end": u.end,
                "text": u.text
            }
            for u in transcript.utterances
        ]
    }
    
    with open("C:\\Вписываем путь до папки выгрузки\\output\\transcript_str_2.json", "w", encoding="utf-8") as f:
        json.dump(structured_data, f, ensure_ascii=False, indent=2)
    
    print("✅ Структурированный JSON сохранен в output/transcript_str_2.json")